In [2]:
from langchain_ollama import ChatOllama
from langchain.tools import tool
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain.agents import create_agent
model = ChatOllama(model='qwen2.5:3b')
@tool
def weather_search(location:str)->str:
    """Get weather forecast for a location."""
    return f"It's cloudy in {location} with temperatures between 23 degree and 11 degree celsius."
model_with_tools = model.bind_tools([weather_search])
messages = [{'role':'user','content':"What's the weather like in Patna?"}]
response = model_with_tools.invoke(messages)
# response = model_with_tools.invoke("What's the weather like in Patna?")
# response
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")


Tool: weather_search
args: {'location': 'Patna'}


In [4]:
messages.append(response)
for tool_call in response.tool_calls:
    if tool_call['name'] == 'weather_search':
        tool_result = weather_search.invoke(tool_call)
    messages.append(tool_result)

print(messages)
final_res = model_with_tools.invoke(messages)
final_res.content

[{'role': 'user', 'content': "What's the weather like in Patna?"}, AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-23T15:45:58.5870156Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11152165100, 'load_duration': 2839932900, 'prompt_eval_count': 154, 'prompt_eval_duration': 6289346100, 'eval_count': 21, 'eval_duration': 1995109000, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019beb88-7549-76f1-ba76-89faa199f1df-0', tool_calls=[{'name': 'weather_search', 'args': {'location': 'Patna'}, 'id': '4bebe98e-113b-4e81-adb3-738c5e7c4c29', 'type': 'tool_call'}], usage_metadata={'input_tokens': 154, 'output_tokens': 21, 'total_tokens': 175}), ToolMessage(content="It's cloudy in Patna with temperatures between 23 degree and 11 degree celsius.", name='weather_search', tool_call_id='4bebe98e-113b-4e81-adb3-738c5e7c4c29')]


'The weather in Patna is currently described as cloudy. The temperature ranges from 23 degrees Celsius during the day to 11 degrees Celsius at night.'

In [5]:
# @tool("population_search")
@tool
def get_population(location:str)->str:
    """Get population of a location."""
    return f"..."
mess = [{'role':'user', 'content':'What is the population of Pune?'}]
model_with_tools = model.bind_tools([weather_search, get_population])
result = model_with_tools.invoke(mess)
for tool_call in result.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"args: {tool_call['args']}")

Tool: get_population
args: {'location': 'Pune'}


In [6]:
from langchain.tools import ToolRuntime
def summarize_conversation(runtime:ToolRuntime)->str:
    """Summarize the conversation so far."""
    messages = runtime.state['messages']
    human_mess = sum(1 for m in messages if m.__class__.__name__ == 'HumanMessage')
    ai_mess = sum(1 for m in messages if m.__class__.__name__ == 'AIMessage')
    tool_mess = sum(1 for m in messages if m.__class__.__name__ == 'ToolMessage')
    return f"Conversation has {human_mess} user messages, {ai_mess} ai responses adn {tool_mess} tool messages"

# model_with_tools = model.bind_tools([summarize_conversation])
messages = [
    {'role':'user', 'content':'Hello'},
    {'role':'assistant','content':'Hello, How may I help you?'},
    {'role':'user','content':'Which language model are you?'},
    {'role':'assistant','content':'I am Qwen2.5:3b provided to you through Ollama.'},
    {'role':'user','content':'Summarize the conversation so far'}
]
agent=create_agent(
    model=model,
    tools=([summarize_conversation]),
    system_prompt='You are a helpful assistant. Answer to user queries with the help of tools provided.'
)
res = agent.invoke({
    'messages':messages
})
res['messages']

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='c2cceb3d-4083-40af-9a59-a7a64a1d979e'),
 AIMessage(content='Hello, How may I help you?', additional_kwargs={}, response_metadata={}, id='0e440e84-f443-4c42-9f66-49dcc2ec6ad1'),
 HumanMessage(content='Which language model are you?', additional_kwargs={}, response_metadata={}, id='e6a93929-508a-4435-b918-dd30d4871750'),
 AIMessage(content='I am Qwen2.5:3b provided to you through Ollama.', additional_kwargs={}, response_metadata={}, id='411d6fc1-0ec3-4e6a-af2c-8d4a9d2cc651'),
 HumanMessage(content='Summarize the conversation so far', additional_kwargs={}, response_metadata={}, id='c27467dd-eb25-4106-bce0-48287af759ff'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-23T15:46:24.5541959Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11135260400, 'load_duration': 104291700, 'prompt_eval_count': 202, 'prompt_eval_duration': 9245261300, 'eval

In [7]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.tools import ToolRuntime

@tool
def get_user_info(user_id:str, runtime:ToolRuntime)->str:
    """Get information about users."""
    store = runtime.store
    user_info = store.get(('users',), user_id)
    return str(user_info.value) if user_info else "Unknown User"

@tool
def set_user_info(user_id:str, user_info:dict[str, Any], runtime:ToolRuntime)->str:
    """Save information of the user."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Saved user succesfully"

agent = create_agent(
    model=model,
    tools=([get_user_info, set_user_info]),
    store=InMemoryStore()
)
agent.invoke({
    'messages':[{'role':'user', 'content':'Save this user information:user_id:me123, name:me, age:20, email:me@vscode.com'}]
    })


{'messages': [HumanMessage(content='Save this user information:user_id:me123, name:me, age:20, email:me@vscode.com', additional_kwargs={}, response_metadata={}, id='c2e3b4be-668a-4e6a-8032-5dd44557ea95'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-23T15:46:46.9490555Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15497406300, 'load_duration': 107449400, 'prompt_eval_count': 224, 'prompt_eval_duration': 10174511200, 'eval_count': 53, 'eval_duration': 5154958800, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019beb89-219a-7273-892c-f491f36e3630-0', tool_calls=[{'name': 'set_user_info', 'args': {'user_id': 'me123', 'user_info': {'age': 20, 'email': 'me@vscode.com', 'name': 'me'}}, 'id': 'e8c39726-d080-4573-8110-57cf3eb48431', 'type': 'tool_call'}], usage_metadata={'input_tokens': 224, 'output_tokens': 53, 'total_tokens': 277}),
  ToolMessage(content='Saved user succesfully', name='set_use

In [8]:
ress = agent.invoke({
    'messages':[{'role':'user', 'content':'Find user with user_id:me123'}]
})
ress['messages'][-1].content

'I found the user with the ID me123. Here are their details:\n- Name: me\n- Email: me@vscode.com\n- Age: 20\n\nIs there anything else you need to know?'

In [9]:
from dataclasses import dataclass
user_database = {
    "user123":{
        'name':'Chandler Bing',
        'account_type':'premium',
        'balance':5000, 
        'email':'chandler@bing.com'
    },
    'user234':{
        'name':'Ross Geller',
        'account_type':'standard',
        'balance':2000,
        'email':'ross@gelller.com'
    }
}
@dataclass
class UserContext:
    user_id:str

@tool 
def get_account_info(runtime:ToolRuntime[UserContext])->str:
    """Get account information of current user."""
    user_id = runtime.context.user_id
    if user_id in user_database:
        user = user_database[user_id]
        return f"Account Holder: {user['name']}\nAccount Type:{user['account_type']}\nBalance:{user['balance']}\nEmail:{user['email']}"
    return "user_not_found"

agent = create_agent(
    model=model,
    tools=([get_account_info]),
    context_schema=UserContext,
    system_prompt = 'You are a financial assistant.'
)
result=agent.invoke({
    'messages':[{'role':'user', 'content':'Show my account infomation'}]},
    context = UserContext(user_id='user123')
)
result['messages'][-1].pretty_print()

================================== Ai Message ==================================

Here is your account information:

- Account Holder: Chandler Bing  
- Account Type: Premium  
- Balance: $5,000  
- Email: chandler@bing.com  

Is there anything else you would like to know?


In [11]:
@tool
def get_weather(location:str,runtime:ToolRuntime)->str:
    """Get weather for a given city."""
    writer = runtime.stream_writer
    writer(f"Fetching weather data for {location}")
    writer(f"Fetched data for {location}")
    return f"It's always sunny in {location}"
agent = create_agent(
    model=model,
    tools=([get_weather]),
    system_prompt='You are a weather assistant.'
)
resp = agent.invoke({
    'messages':[{'role':'user','content':'What is the weather for Patna?'}]
})
resp['messages']

[HumanMessage(content='What is the weather for Patna?', additional_kwargs={}, response_metadata={}, id='6c4f4f2c-0b86-4c4f-8450-4cc4e514dcf0'),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-01-23T15:50:39.440635Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2359502700, 'load_duration': 144606000, 'prompt_eval_count': 143, 'prompt_eval_duration': 124223700, 'eval_count': 21, 'eval_duration': 2064598700, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019beb8c-e116-7fa2-996c-623ea7808c5b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Patna'}, 'id': '9e305b3c-8582-40d6-b54f-7e8933d71bf8', 'type': 'tool_call'}], usage_metadata={'input_tokens': 143, 'output_tokens': 21, 'total_tokens': 164}),
 ToolMessage(content="It's always sunny in Patna", name='get_weather', id='09d6c3f5-e3a7-4f58-8e2c-867b9cf44c7d', tool_call_id='9e305b3c-8582-40d6-b54f-7e8933d71bf8'),
 AIMessage(content='The we